# Nano-Constellation

Finding particles in an AFM image of a TiO₂ thin film, joining them into a mesh, and looking at how the spacing between them varies.

Built during the UCD Interdisciplinary Hackathon (December 2025) by Sinny J. Trivedi. The TiO₂ film is from my PhD work.

**Steps**
1. Load and normalise the image
2. Crop away the noisy edges
3. A quick look: bright regions above a threshold
4. Find individual particles (`peak_local_max`)
5. Join them into a mesh (Delaunay triangulation)
6. Colour each connection by its length (spacing map)
7. View the surface and particles in 3D
8. Summary numbers and CSV export

**Note:** this works on an exported AFM *image*, so brightness stands in for height. It is an exploration, not a calibrated measurement.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.collections as mc
from skimage import io, color, exposure
from skimage.feature import peak_local_max
from scipy.spatial import Delaunay

# Settings: everything you might want to change is here
IMAGE_PATH = "TiO2_film.png"

CROP = {"top": 0.05, "bottom": 0.10, "left": 0.05, "right": 0.15}  # fraction removed from each side
THRESHOLD = 0.5        # step 3: brightness cut-off for the quick look
MIN_DISTANCE = 15      # step 4: smallest allowed gap between two particles (pixels)
THRESHOLD_REL = 0.2    # step 4: ignore peaks dimmer than 20% of the brightest one

## 1. Load and normalise
Convert to grayscale and scale every pixel to the range 0 to 1.

In [ ]:
img_raw = io.imread(IMAGE_PATH)

if img_raw.ndim == 3:
    img = color.rgb2gray(img_raw[:, :, :3])   # drop the alpha channel if there is one
else:
    img = img_raw.astype(float)

img_norm = (img - img.min()) / (img.max() - img.min())
print("Image size (pixels):", img_norm.shape)

## 2. Crop the edges
The right-hand edge of the scan has a bright saturated band, so more is cut from that side. Then the contrast is stretched across the remaining area.

In [ ]:
h, w = img_norm.shape
t, b = int(h * CROP["top"]), int(h * CROP["bottom"])
l, r = int(w * CROP["left"]), int(w * CROP["right"])

img_clean = img_norm[t:h - b, l:w - r]
img_clean = exposure.rescale_intensity(img_clean, out_range=(0, 1))
print("Size after cropping:", img_clean.shape)

## 3. Quick look: bright regions
A simple threshold shows *where* the bright areas are. This is panel (b) of the summary figure. It is not the same as finding individual particles, which happens in step 4.

In [ ]:
bright = img_clean > THRESHOLD

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_clean, cmap="gray"); axes[0].set_title("(a) Normalised scan")
axes[1].imshow(bright, cmap="gray");    axes[1].set_title(f"(b) Regions above {THRESHOLD:.0%} brightness")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Find individual particles
`peak_local_max` finds the local brightest point of each particle. `MIN_DISTANCE` stops one particle being counted twice.

In [ ]:
coords = peak_local_max(img_clean, min_distance=MIN_DISTANCE, threshold_rel=THRESHOLD_REL)
print(f"Particles found: {len(coords)}")

plt.figure(figsize=(7, 7))
plt.imshow(img_clean, cmap="magma")
plt.scatter(coords[:, 1], coords[:, 0], s=15, c="white", edgecolors="cyan")
plt.title(f"Detected particles ({len(coords)})"); plt.axis("off"); plt.show()

## 5. Join them into a mesh (the constellation)
Delaunay triangulation connects each particle to its natural neighbours, so that no other particle sits inside the circle through any triangle's three corners.

In [ ]:
tri = Delaunay(coords)

plt.figure(figsize=(7, 7))
plt.imshow(img_clean, cmap="magma")
plt.triplot(coords[:, 1], coords[:, 0], tri.simplices, color="cyan", lw=0.8, alpha=0.6)
plt.scatter(coords[:, 1], coords[:, 0], s=15, c="white", edgecolors="cyan", zorder=5)
plt.title("(c) Delaunay mesh"); plt.axis("off"); plt.show()

## 6. Spacing map
Each connection is coloured by its length: blue for particles close together, red for far apart.

Triangles along the outer edge of the image are often long and thin, so they are flagged and left out of the statistics.

In [ ]:
# Unique edges of the mesh
edges = set()
for simplex in tri.simplices:
    for i in range(3):
        edges.add(tuple(sorted((simplex[i], simplex[(i + 1) % 3]))))
edges = np.array(sorted(edges))

lengths = np.linalg.norm(coords[edges[:, 0]] - coords[edges[:, 1]], axis=1)

# Edges on the outer boundary of the mesh
hull_edges = {tuple(sorted(e)) for e in tri.convex_hull}
is_inner = np.array([tuple(e) not in hull_edges for e in edges])

lines = [[(coords[a, 1], coords[a, 0]), (coords[b, 1], coords[b, 0])] for a, b in edges[is_inner]]

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(img_clean, cmap="gray", alpha=0.5)
lc = mc.LineCollection(lines, cmap="coolwarm", linewidths=1.5)
lc.set_array(lengths[is_inner])
ax.add_collection(lc)
ax.scatter(coords[:, 1], coords[:, 0], s=10, c="black", zorder=5)
fig.colorbar(lc, ax=ax, fraction=0.046, pad=0.04, label="Distance between neighbours (pixels)")
ax.set_title("Spacing between neighbouring particles"); ax.axis("off"); plt.show()

## 7. 3D view
The image brightness is drawn as height, with each particle floating just above its peak.

In [ ]:
yy, xx = np.indices(img_clean.shape)
z = img_clean[coords[:, 0], coords[:, 1]]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(xx, yy, img_clean, cmap="magma", edgecolor="none", alpha=0.8, rstride=2, cstride=2)
ax.scatter(coords[:, 1], coords[:, 0], z + 0.05, c="cyan", s=15, edgecolors="white")
ax.view_init(elev=45, azim=45)
ax.set_title("(d) 3D view"); ax.set_axis_off(); plt.show()

## 8. Summary and export

In [ ]:
inner = lengths[is_inner]
print(f"Particles found:                {len(coords)}")
print(f"Neighbour connections (inner):  {len(inner)}")
print(f"Median spacing:                 {np.median(inner):.1f} px")
print(f"Spread (interquartile range):   {np.percentile(inner, 25):.1f} to {np.percentile(inner, 75):.1f} px")

pd.DataFrame(coords, columns=["y_px", "x_px"]).to_csv("particle_coordinates.csv", index=False)
print("Saved particle_coordinates.csv")

## Limitations and next steps
- Brightness in an exported image stands in for height; using the raw AFM height data would make this quantitative.
- Distances are in pixels. Converting to nanometres needs the scan's pixel size.
- The settings in step 0 change how many particles are found, so results should be read alongside them.
- A natural next step would be to compare the spacing map with other measurements of the same film.